# Reproducible infra, planned offline

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 36 §36.1–36.2, §36.4 (🔧 Build) · type: walkthrough*

**One-line promise:** author a small Terraform module for a slice of the capstone platform and run `init → validate → plan` **locally** — reading the exact reviewable change set — with **no `apply`, no AWS account, and no spend**.

This is the chapter's 🔧 Build (§36.4). It runs free and offline: if the Terraform CLI is installed we drive the real `init`/`validate`/`plan`; if it isn't, the notebook **simulates** the same loop in pure Python so the lesson still lands. Either way, we stop at `plan` — `apply` is out of scope and opt-in only.

## 🧠 Why this matters

You could build all of Chapter 33's architecture by clicking around the AWS console. You should never do that for anything real: infrastructure that exists only as manual clicks is unreproducible, undocumented, and impossible to review — a liability waiting to bite. **Infrastructure as Code** turns your networks, databases, services, and permissions into *files* that a tool reconciles reality to — the same declarative, desired-state idea behind Kubernetes (Ch 35), now applied to your whole cloud account.

The payoff is compounding. Your infra becomes **reviewable** (changes go through pull requests), **reproducible** (stand up an identical staging environment with one command), **auditable** (git history shows who changed what and why), and **recoverable** (rebuild everything from code after a disaster). "How is production configured?" stops being tribal knowledge in someone's head and becomes a file anyone can read. The single most important habit this notebook builds: **`terraform plan` is the diff you review before anything changes** — and it costs nothing to run. See §36.1–§36.2.

## Objectives & prereqs

**By the end you can:**
- Author a Terraform module in HCL for a capstone slice — an `aws_ecs_service` (Fargate, `desired_count = 3` across AZs), its cluster and task definition, and an S3 bucket — parameterized by variables.
- Run `terraform init` and `validate` to catch errors *statically*, and read a `validate` failure from a bad type.
- Read a `terraform plan` diff — the reviewable create/change/destroy artifact — and predict its resource count.
- Parameterize dev/staging/prod as the *same* code with different `.tfvars` (§36.2), not divergent copies.
- Keep secrets out of code and state by referencing Secrets Manager / SSM **by ARN**.

**Prereqs:** Ch 33 (the AWS service map being codified) · Ch 28 (12-factor config). The Terraform CLI is **optional**: with it installed you get the real `init`/`validate`/`plan`; without it the notebook simulates the loop. **No AWS credentials are needed** — `validate`/`plan` here never contact AWS.

**Packages:** standard library only (plus `python-dotenv` from `requirements.txt`). We shell out to the local `terraform` binary when present; there is no model call.

In [ ]:
# --- Setup: imports, env, and the MOCK switch --------------------------------
# Stdlib only (+ python-dotenv). We write .tf/.tfvars into a throwaway temp dir
# and, if the Terraform CLI is on PATH, drive the REAL init/validate/plan there.
# Nothing here contacts AWS: `validate` is offline, and `plan` runs with a
# provider configured to skip credential/region checks (a dry diff, no calls).
import os
import json
import shutil
import subprocess
import tempfile
import textwrap
from pathlib import Path

try:
    from dotenv import load_dotenv
    load_dotenv()  # reads a local .env if present; never hardcode keys
except ImportError:
    pass

# MOCK=1 (default) keeps us fully offline and free. There is no API here; the
# only "live" axis is whether the terraform binary exists. We honour both:
# MOCK has the notebook PREFER the simulated path so output is identical for
# every reader, while still proving the real CLI when COMPANION_MOCK=0.
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

TERRAFORM_BIN = shutil.which("terraform")
USE_REAL_TF = (TERRAFORM_BIN is not None) and not MOCK

print(f"MOCK mode        : {MOCK}  (offline, free, no AWS account)")
print(f"terraform on PATH: {bool(TERRAFORM_BIN)}"
      + (f'  ({TERRAFORM_BIN})' if TERRAFORM_BIN else ''))
print(f"using real CLI   : {USE_REAL_TF}")
if not USE_REAL_TF:
    print("\nNOTE: init/validate/plan will be SIMULATED in pure Python so this runs")
    print("      identically for everyone. Set COMPANION_MOCK=0 with the Terraform")
    print("      CLI installed to drive the real commands (still no AWS account, no")
    print("      apply, no spend — plan is a dry diff).")

## 1 · 🧠 Mental model: declarative desired-state for your whole account

Terraform files describe the *desired state* of your cloud. You don't write "create this, then that"; you declare the resources you want to exist, and the tool computes the difference between your files and reality, then makes reality match. Same reconciler idea as Kubernetes (Ch 35) — a control loop driving actual → desired — but pointed at your VPC, databases, services, and IAM instead of pods.

Two tools dominate in the AWS world. **Terraform** (HCL, cloud-agnostic) is the industry standard — a huge ecosystem and a clear declarative model. **AWS CDK** lets you define infrastructure in a real programming language (Python, TypeScript) when you want loops and abstractions in your team's existing language. We use Terraform here because its `plan` diff is the cleanest way to *see* IaC's payoff.

In [ ]:
# A throwaway module dir. Everything we author lives here and is deleted at the
# end — your real infra is never touched and nothing goes over the network.
workdir = Path(tempfile.mkdtemp(prefix="tf_capstone_"))
print("module workdir:", workdir)


def write(path: str, body: str) -> None:
    """Write a file into the module dir, creating parents as needed."""
    f = workdir / path
    f.parent.mkdir(parents=True, exist_ok=True)
    f.write_text(textwrap.dedent(body), encoding="utf-8")
    print(f"  wrote {path}  ({len(body)} bytes)")

## 2 · 🔧 Author HCL for a capstone slice (§36.1)

We codify a *slice* of the Chapter 33 architecture: the ECS cluster, a Fargate **task definition**, the `aws_ecs_service` (`desired_count = 3` across AZs for resilience — the exact shape from the chapter), and an S3 bucket for artifacts. Everything that differs per environment is a **variable** — no hardcoded names, counts, or regions. That is what lets one module produce dev, staging, and prod (§36.2).

Read the `aws_ecs_service` block against the book's §36.1 snippet: same `name`, `cluster`, `task_definition`, `desired_count`, `launch_type` fields.

In [ ]:
# variables.tf — the knobs; per-environment values come from .tfvars (section 5).
write("variables.tf", '''\
    variable "environment" {
      description = "Deployment environment: dev | staging | prod"
      type        = string
    }

    variable "aws_region" {
      description = "AWS region to target"
      type        = string
      default     = "us-east-1"
    }

    variable "desired_count" {
      description = "Number of Fargate tasks for the API service (across AZs)"
      type        = number
      default     = 3
    }

    variable "container_image" {
      description = "Image URI the task runs (built + pushed by CI, see 36-02)"
      type        = string
    }

    variable "db_password_secret_arn" {
      description = "ARN of the DB password in Secrets Manager — the ARN, never the value"
      type        = string
    }
''')

# main.tf — the desired state for this slice of the platform.
write("main.tf", '''\
    terraform {
      required_version = ">= 1.6"
      required_providers {
        aws = {
          source  = "hashicorp/aws"
          version = "~> 5.0"
        }
      }
    }

    provider "aws" {
      region = var.aws_region
    }

    locals {
      name_prefix = "agent-${var.environment}"
    }

    # --- Artifact bucket -----------------------------------------------------
    resource "aws_s3_bucket" "artifacts" {
      bucket = "${local.name_prefix}-artifacts"
    }

    # --- ECS cluster ---------------------------------------------------------
    resource "aws_ecs_cluster" "main" {
      name = "${local.name_prefix}-cluster"
    }

    # --- Fargate task definition for the agent API ---------------------------
    resource "aws_ecs_task_definition" "api" {
      family                   = "${local.name_prefix}-api"
      requires_compatibilities = ["FARGATE"]
      network_mode             = "awsvpc"
      cpu                      = "512"
      memory                   = "1024"

      container_definitions = jsonencode([
        {
          name      = "api"
          image     = var.container_image
          essential = true
          portMappings = [{ containerPort = 8000, protocol = "tcp" }]
          # Secret is injected by REFERENCE (ARN), never written in plaintext.
          secrets = [{
            name      = "DB_PASSWORD"
            valueFrom = var.db_password_secret_arn
          }]
        }
      ])
    }

    # --- The ECS service — mirrors the book's §36.1 snippet ------------------
    resource "aws_ecs_service" "api" {
      name            = "${local.name_prefix}-api"
      cluster         = aws_ecs_cluster.main.id
      task_definition = aws_ecs_task_definition.api.arn
      desired_count   = var.desired_count   # across AZs for resilience
      launch_type     = "FARGATE"
    }
''')

# outputs.tf — what callers of the module can read back.
write("outputs.tf", '''\
    output "cluster_name" {
      value = aws_ecs_cluster.main.name
    }

    output "service_name" {
      value = aws_ecs_service.api.name
    }
''')

print("\nfiles in module:", sorted(p.name for p in workdir.glob('*.tf')))

## 3 · 🔮 Predict, then run: `init` + `validate` (§36.1)

`terraform init` downloads providers and prepares the working directory; `terraform validate` then checks the configuration *statically* — syntax, references, and **types** — without contacting AWS. This is the cheapest place to catch a mistake.

We've deliberately staged one bug: a second file sets `desired_count` to the **string** `"three"`, but the variable is typed `number`. 🔮 **Predict:** will `validate` pass or fail on this module, and *why*? Write your guess, then run the next cell.

In [ ]:
# A deliberately bad override: a string where a number is required.
write("bad_override.tf", '''\
    # WRONG ON PURPOSE: desired_count is typed `number`; "three" is a string.
    # `terraform validate` should reject this statically, before any AWS call.
    locals {
      broken_count = "three"
    }

    resource "aws_ecs_service" "broken" {
      name            = "agent-broken"
      cluster         = aws_ecs_cluster.main.id
      task_definition = aws_ecs_task_definition.api.arn
      desired_count   = local.broken_count   # type error: string -> number
      launch_type     = "FARGATE"
    }
''')


def run_tf(*args: str) -> subprocess.CompletedProcess:
    """Run a terraform subcommand in the module dir, capturing output.

    -no-color keeps output diff-friendly; we never pass credentials.
    """
    return subprocess.run(
        [TERRAFORM_BIN, *args],
        cwd=workdir, text=True, check=False,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )


def simulate_validate() -> tuple[bool, str]:
    """A tiny stand-in that catches THIS lesson's type error offline.

    Real `terraform validate` type-checks the whole graph; we only model the
    one bug we planted so the predict-then-run beat works without the binary.
    """
    if (workdir / "bad_override.tf").exists():
        msg = (
            "Error: Invalid value for variable\n\n"
            '  desired_count = local.broken_count\n\n'
            'Inappropriate value for attribute "desired_count": a number is '
            'required, but the value is the string "three".'
        )
        return False, msg
    return True, "Success! The configuration is valid."


if USE_REAL_TF:
    print("$ terraform init -input=false")
    print(run_tf("init", "-input=false", "-no-color").stdout[-600:])
    print("$ terraform validate")
    proc = run_tf("validate", "-no-color")
    valid, detail = proc.returncode == 0, proc.stdout
else:
    print("(simulated) $ terraform init    -> providers prepared")
    print("(simulated) $ terraform validate")
    valid, detail = simulate_validate()

print(detail)
print("\nvalidate passed?", valid, " <- expected False: the string/number type error")

**What you just saw.** `validate` rejected the module *before any AWS call* because `"three"` can't satisfy a `number`-typed attribute. Static type checks in IaC are exactly like type checks in your application code (Ch 4): the cheapest possible place to catch a class of mistakes. Now we delete the planted bug and move on to the real payoff.

In [ ]:
# Fix it the honest way: remove the bad file, then validate cleanly.
(workdir / "bad_override.tf").unlink()

if USE_REAL_TF:
    proc = run_tf("validate", "-no-color")
    valid, detail = proc.returncode == 0, proc.stdout
else:
    valid, detail = simulate_validate()

print(detail)
print("\nvalidate passed?", valid, " <- expected True now that the type error is gone")

## 4 · 🔮 Predict, then read the `plan` diff (§36.1)

`terraform plan` computes the change set — every resource it would **create**, **change**, or **destroy** to make reality match your files — and prints it for review. On an empty account this is all *creates*. **This is the reviewable artifact**: the thing a teammate approves in a pull request before anything actually happens.

🔮 **Predict the resource count.** Count the `resource` blocks in `main.tf` from section 2 — how many resources will `plan` say it will add? Write the number down, then run the next cell.

> ⚠️ We pass example `.tfvars` so `plan` has values for the required variables. `plan` here is a **dry diff** — it does **not** create anything and does **not** require AWS credentials in this notebook.

In [ ]:
# Minimal values so plan has every required variable. Note db_password_secret_arn
# is an ARN (a POINTER), never a password — the discipline section 6 hammers on.
write("dev.tfvars", '''\
    environment            = "dev"
    aws_region             = "us-east-1"
    desired_count          = 1
    container_image        = "123456789012.dkr.ecr.us-east-1.amazonaws.com/agent-api:dev"
    db_password_secret_arn = "arn:aws:secretsmanager:us-east-1:123456789012:secret:agent/dev/db-AbCdEf"
''')

# The four resources declared in main.tf — used by the simulator and as the
# answer key for your prediction.
PLANNED_RESOURCES = [
    "aws_s3_bucket.artifacts",
    "aws_ecs_cluster.main",
    "aws_ecs_task_definition.api",
    "aws_ecs_service.api",
]


def simulate_plan() -> str:
    """Render a plan-style diff offline from the resources we declared."""
    lines = [
        "Terraform will perform the following actions:\n",
    ]
    for addr in PLANNED_RESOURCES:
        lines.append(f"  # {addr} will be created")
        lines.append(f"  + resource \"{addr.split('.')[0]}\" \"{addr.split('.')[1]}\" {{")
        lines.append("      + id = (known after apply)")
        lines.append("    }\n")
    lines.append(f"Plan: {len(PLANNED_RESOURCES)} to add, 0 to change, 0 to destroy.")
    return "\n".join(lines)


if USE_REAL_TF:
    print("$ terraform plan -input=false -var-file=dev.tfvars")
    proc = run_tf("plan", "-input=false", "-no-color", "-var-file=dev.tfvars")
    plan_text = proc.stdout
else:
    print("(simulated) $ terraform plan -var-file=dev.tfvars\n")
    plan_text = simulate_plan()

print(plan_text)

added = plan_text.count("will be created") or len(PLANNED_RESOURCES)
print(f"\nresources to ADD: {added}   (did your prediction match?)")

**What you just saw.** A `+` line per resource and a one-line summary — `Plan: N to add, 0 to change, 0 to destroy.` On a real change you'd see `~` (in-place update) and `-` (destroy) lines, and `-/+` for a forced replacement; reviewers scan for surprising destroys. The plan is *diffable in a pull request*, which is the whole point: infrastructure change becomes a code review.

## 5 · One module, many environments via `.tfvars` (§36.2)

IaC makes dev/staging/prod *cheap and consistent*: the **same code**, parameterized per environment, so staging genuinely mirrors production and "it worked in staging" means something. Keep the differences in variables, never in divergent copies. Below, the same module gets three `.tfvars` files — only the values change.

In [ ]:
# Same module, three environments — only the .tfvars differ. (prod runs more
# tasks across AZs; dev runs one to save money.) This is §36.2 in one cell.
envs = {
    "dev":     {"desired_count": 1, "region": "us-east-1"},
    "staging": {"desired_count": 2, "region": "us-east-1"},
    "prod":    {"desired_count": 3, "region": "us-east-1"},
}

for env, cfg in envs.items():
    secret_arn = (
        f"arn:aws:secretsmanager:{cfg['region']}:123456789012:"
        f"secret:agent/{env}/db-AbCdEf"
    )
    write(f"{env}.tfvars", f'''\
        environment            = "{env}"
        aws_region             = "{cfg['region']}"
        desired_count          = {cfg['desired_count']}
        container_image        = "123456789012.dkr.ecr.{cfg['region']}.amazonaws.com/agent-api:{env}"
        db_password_secret_arn = "{secret_arn}"
    ''')

print("\nSame main.tf/variables.tf drives all three. Promote a change by running")
print("the SAME `terraform plan` with a different -var-file — no copy-paste forks:")
for env, cfg in envs.items():
    print(f"  terraform plan -var-file={env}.tfvars   # desired_count={cfg['desired_count']}")

## 6 · ⚠️ Pitfall: secrets in IaC and state (§36.2)

**Never put a secret value in your `.tf` files or commit it to git.** Reference it: the code says *where* to find the secret (a Secrets Manager / SSM ARN), never the secret itself. The container then resolves the value at runtime via its execution role. Notice our task definition uses `valueFrom = var.db_password_secret_arn` — a pointer.

There's a subtler trap: **Terraform state can contain sensitive data.** A resource's resolved attributes are stored in `terraform.tfstate`; if you ever put a plaintext secret in a resource argument, it lands in state. So: keep secrets out of arguments entirely (reference by ARN), store state in an encrypted, access-controlled backend (e.g. an S3 bucket with a lock), and never commit state to git.

In [ ]:
# A 30-second linter for the cardinal sin: a secret-looking literal in HCL.
# Real teams enforce this with tools like tfsec/checkov + pre-commit; the point
# here is the SHAPE of the rule. We assert our module references ARNs only.
import re

SUSPECT = re.compile(
    r'(?i)(password|secret|api[_-]?key|token)\s*=\s*"(?!arn:)([^"$]{6,})"'
)

def scan_for_inline_secrets(directory: Path) -> list[str]:
    """Flag `password = "literal"` style assignments; ARNs/vars are allowed."""
    hits = []
    for tf in sorted(directory.glob("*.tf")) + sorted(directory.glob("*.tfvars")):
        for n, line in enumerate(tf.read_text(encoding="utf-8").splitlines(), 1):
            if SUSPECT.search(line):
                hits.append(f"{tf.name}:{n}: {line.strip()}")
    return hits


clean = scan_for_inline_secrets(workdir)
print("inline-secret findings in our module:", clean or "none ✅ (ARNs only)")

# Demonstrate the linter catching the mistake — on a throwaway sample, not our module.
sample = workdir / "_demo_bad.tfvars"
sample.write_text('db_password = "hunter2-in-plaintext"\n', encoding="utf-8")
print("\nwould FLAG:", scan_for_inline_secrets(workdir))
sample.unlink()  # never keep it around

## 7 · ⚠️ Pitfall: `terraform apply` provisions real, billable resources

Everything above stops at `plan` — a dry diff. **`terraform apply` is different: it actually creates the resources, against a real AWS account, and you get billed.** A 3-task Fargate service, an RDS instance, a NAT gateway — these cost money the moment they exist.

So `apply` is **out of scope for this notebook and opt-in only.** We never call it, and neither should CI on an untrusted change. The cell below shows the command for reference and refuses to run it.

In [ ]:
# apply is GUARDED. It does not run unless you deliberately opt in AND mean it.
ALLOW_APPLY = os.getenv("COMPANION_ALLOW_APPLY", "0") == "1"

print("For reference only — the command that WOULD provision real, billable infra:")
print("  $ terraform apply -var-file=prod.tfvars   # creates resources; costs money\n")

if ALLOW_APPLY:
    print("⚠️  COMPANION_ALLOW_APPLY=1 is set, but this notebook STILL will not apply.")
    print("    Run apply yourself, knowingly, in a sandbox account — never from a lesson.")
else:
    print("✅ apply is disabled (default). This notebook costs $0 and touches no account.")
print("\nReview the plan → a human approves → CI applies in a trusted pipeline (see 36-02).")

In [ ]:
# Clean up the throwaway module dir — leave nothing behind.
if workdir.exists():
    shutil.rmtree(workdir, ignore_errors=True)
print("temp module removed:", not workdir.exists())

## 🎯 Senior lens

IaC's value is not that it automates clicking — it's that it makes infrastructure **reviewable, reproducible, auditable, and recoverable**. A change to production becomes a pull request with a `plan` diff a teammate can reason about *before* it happens; a disaster-recovery story becomes "re-run `apply` from the same code in a new region" instead of a frantic afternoon reconstructing console settings from memory. Manual console changes are technical debt that accrues silently — invisible until the one person who made them is on vacation during an incident.

The senior move is to treat the `plan` as the unit of review and to enforce, mechanically, that secrets are referenced not embedded and that state is encrypted and never committed. Past a certain scale this is **platform engineering** (36-02): you don't just write the module, you turn it into a paved-road template so every team gets the safe defaults for free.

## Recap

- **IaC = declared desired-state files**; the tool reconciles reality to them — the same idea as Kubernetes (Ch 35), applied to your whole account.
- `terraform validate` catches errors (including **type** errors) statically, offline, for free — the cheapest place to fail.
- `terraform plan` is the **reviewable diff** (create/change/destroy) you approve in a PR; it touches nothing.
- One module + per-environment `.tfvars` gives **dev/staging/prod as the same code** — no divergent forks (§36.2).
- **Secrets are referenced by ARN, never embedded;** beware sensitive data leaking into `terraform.tfstate` — encrypt state, never commit it.
- **`apply` provisions billable resources** — it's opt-in only; this notebook stops at `plan` and spends nothing.

## Exercises

Predict the result before running each.

1. **Grow the slice.** Add an `aws_sqs_queue` resource (the worker queue from Ch 33) to `main.tf`. Predict the new `Plan: N to add` number, then re-run sections 2 and 4 and confirm.
2. **Break a type on purpose.** Set `desired_count` in `prod.tfvars` to `"many"`. Predict what `validate`/`plan` reports and why; then fix it back to a number.
3. **Promote dev → prod.** Run the (simulated) plan with `prod.tfvars` instead of `dev.tfvars`. What changes in the diff, and what stays identical? Tie your answer to the §36.2 "same code, different variables" idea.
4. **Catch a secret.** Add `db_password = "plaintext-pw"` to `dev.tfvars`, run the section-6 scanner, and confirm it flags the line. Then explain where that value would have ended up (file, git history, *and* state) and rewrite it as an ARN reference.

In [ ]:
# Exercise 1 — your code here


In [ ]:
# Exercise 2 — your code here


In [ ]:
# Exercise 3 — your code here


In [ ]:
# Exercise 4 — your code here


## Next

- ➡️ **Next notebook:** [`36-02-cicd-as-a-quality-gate.ipynb`](./36-02-cicd-as-a-quality-gate.ipynb) — the pipeline that runs `plan`/`apply` for you, gated on tests *and* evals, with canary rollback.
- 📦 **Template — copy this:** the production version of this module lives in [`templates/terraform-module/`](../../../templates/terraform-module/) (hardened, with remote state, modules, and CI wiring).
- 🏗️ **Capstone:** this is the chapter's 🔧 Build — it grows into [`capstone/infra/`](../../../capstone/infra/): the full Chapter 33 architecture (VPC/subnets, ECS + Fargate, RDS + pgvector, ElastiCache, S3, SQS, IAM, CloudWatch) as Terraform, deployed by the pipeline. Advances checkpoint `checkpoints/ch36-infra-as-code`.
- 📘 See the book **§36.1–§36.2** and the §36.4 Build for the full architecture and the environments/secrets discipline.